# Camada analítica — BCI IV 2a (Parquet + DuckDB)

Esquema estrela sobre o 2a: uma tabela fato **trial-a-trial** e dimensões (sujeito, classe,
sessão, run), em Parquet e consultadas com DuckDB. O desenho do esquema está documentado no
[`../README.md`](../README.md). A ingestão vive em [`../src/ingest.py`](../src/ingest.py)
(constrói as tabelas a partir dos metadados do MOABB, **sem** o sinal bruto).

## Tabelas

As tabelas são geradas por `python -m src.ingest`. A célula abaixo as regera apenas se ainda
não existirem (evita rebaixar o dataset a cada execução).

In [8]:
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, "..")
P = Path("..") / "data" / "processed"

if not (P / "fact_trial.parquet").exists():
    from src.ingest import build_tables, write_tables
    write_tables(build_tables(), P)
    print("tabelas geradas")
else:
    print("tabelas presentes em", P)

con = duckdb.connect()
fact = f"'{P.as_posix()}/fact_trial.parquet'"
dim_class = f"'{P.as_posix()}/dim_class.parquet'"
dim_session = f"'{P.as_posix()}/dim_session.parquet'"
dim_subject = f"'{P.as_posix()}/dim_subject.parquet'"

tabelas presentes em ..\data\processed


## Schema explícito e nulos verdadeiros

Identificadores como texto, tipos definidos, e demografia ausente como **NULL** (não `-1`/`999`).

In [9]:
con.sql(f"DESCRIBE SELECT * FROM {fact}").df()

,column_name,column_type,null,key,default,extra
0,trial_id,VARCHAR,YES,None,None,None
1,subject_id,VARCHAR,YES,None,None,None
2,session_id,VARCHAR,YES,None,None,None
3,run_id,VARCHAR,YES,None,None,None
4,class_id,SMALLINT,YES,None,None,None
5,trial_in_run,SMALLINT,YES,None,None,None
6,onset_s,DOUBLE,YES,None,None,None
7,duration_s,DOUBLE,YES,None,None,None
8,n_samples,INTEGER,YES,None,None,None


In [10]:
# dim_subject: age/sex/handedness são NULL (o 2a não publica demografia)
con.sql(f"SELECT * FROM {dim_subject}").df()

,subject_id,age,sex,handedness
0,A01,<NA>,None,None
1,A02,<NA>,None,None
2,A03,<NA>,None,None
3,A04,<NA>,None,None
4,A05,<NA>,None,None
5,A06,<NA>,None,None
6,A07,<NA>,None,None
7,A08,<NA>,None,None
8,A09,<NA>,None,None


## Consultas (respondem o EDA do 2a)

In [11]:
# Q1 — as 4 classes estão balanceadas em cada papel de sessão?
con.sql(f"""
    SELECT s.session_role, c.class_name, COUNT(*) AS n_trials
    FROM {fact} f
    JOIN {dim_class} c USING (class_id)
    JOIN {dim_session} s USING (session_id)
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

,session_role,class_name,n_trials
0,test,feet,648
1,test,left_hand,648
2,test,right_hand,648
3,test,tongue,648
4,train,feet,648
5,train,left_hand,648
6,train,right_hand,648
7,train,tongue,648


In [12]:
# Q2 — quantos trials por papel de sessão (treino vs teste)?
con.sql(f"""
    SELECT s.session_role, COUNT(*) AS n_trials
    FROM {fact} f
    JOIN {dim_session} s USING (session_id)
    GROUP BY 1
""").df()

,session_role,n_trials
0,train,2592
1,test,2592


In [13]:
# Q3 — cobertura: trials por sujeito
con.sql(f"""
    SELECT subject_id, COUNT(*) AS n_trials
    FROM {fact}
    GROUP BY 1
    ORDER BY 1
""").df()

,subject_id,n_trials
0,A01,576
1,A02,576
2,A03,576
3,A04,576
4,A05,576
5,A06,576
6,A07,576
7,A08,576
8,A09,576


## Benchmark — CSV vs Parquet

Mesma tabela fato nos dois formatos: tamanho em disco e tempo médio de uma agregação.

In [14]:
import os, time

csv_path = (P / "fact_trial.csv").as_posix()
pq_path = (P / "fact_trial.parquet").as_posix()

csv_kib = os.path.getsize(csv_path) / 1024
pq_kib = os.path.getsize(pq_path) / 1024
print(f"CSV:     {csv_kib:7.1f} KiB")
print(f"Parquet: {pq_kib:7.1f} KiB  ({csv_kib / pq_kib:.1f}x menor)")

def bench(query, n=20):
    t = time.perf_counter()
    for _ in range(n):
        con.sql(query).fetchall()
    return (time.perf_counter() - t) / n * 1000

ms_csv = bench(f"SELECT class_id, COUNT(*) FROM read_csv_auto('{csv_path}') GROUP BY 1")
ms_pq = bench(f"SELECT class_id, COUNT(*) FROM read_parquet('{pq_path}') GROUP BY 1")
print(f"\nConsulta CSV:     {ms_csv:6.2f} ms")
print(f"Consulta Parquet: {ms_pq:6.2f} ms  ({ms_csv / ms_pq:.0f}x mais rápido)")

CSV:       299.9 KiB
Parquet:    37.7 KiB  (8.0x menor)

Consulta CSV:      43.06 ms
Consulta Parquet:   1.05 ms  (41x mais rápido)
